# Entropy-Gated Jury — Linux / Colab A100 (vLLM)

Run the full EGJ pipeline on any Linux machine with a CUDA GPU (A100, H100, V100, etc.).

**Runtime checklist (Colab)**
- Runtime → Change runtime type → **A100 GPU** (or T4 for budget tests)
- High-RAM checkbox → on

**Steps**
1. Clone repo & install
2. Authenticate with Hugging Face (needed to pull Qwen3.5 and Gemma 4)
3. Download model weights
4. Switch config to `backend: vllm`
5. Run the pipeline

In [ ]:
# Sanity-check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Step 1 — Clone the repository

In [ ]:
# Edit REPO_URL to point at your fork if needed
REPO_URL = "https://github.com/YOUR_USERNAME/hackaithon-innovator.git"

import os, subprocess
if not os.path.isdir("hackaithon-innovator"):
    subprocess.run(["git", "clone", REPO_URL], check=True)

os.chdir("hackaithon-innovator")
print("CWD:", os.getcwd())

## Step 2 — Install dependencies

In [ ]:
# vLLM ships its own torch; install it first, then the rest
!pip install -q -r requirements-vllm.txt
print("Done installing.")

## Step 3 — Hugging Face authentication

Models are gated. You need an HF token with access to Qwen3.5-9B and Gemma 4 E4B.

In [ ]:
# Option A: notebook secret (recommended on Colab)
# Add HF_TOKEN to Colab Secrets (key icon in the left panel)
try:
    from google.colab import userdata
    import os
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets.")
except Exception:
    # Option B: paste token directly (don't commit this to git)
    import os
    os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"
    print("HF_TOKEN set manually.")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Logged in to Hugging Face.")

## Step 4 — Download model weights

This pulls Qwen3.5-9B (~18 GB) and Gemma 4 E4B (~9 GB) from the Hub.
Expect 5–10 min on Colab depending on bandwidth.

In [ ]:
from huggingface_hub import snapshot_download
import os

os.makedirs("weights", exist_ok=True)

print("Downloading Qwen/Qwen3.5-9B ...")
snapshot_download(
    repo_id="Qwen/Qwen3.5-9B",
    local_dir="weights/Qwen3.5-9B",
    ignore_patterns=["*.bin"],   # prefer safetensors
)
print("Qwen done.")

print("Downloading google/gemma-4-E4B-it ...")
snapshot_download(
    repo_id="google/gemma-4-E4B-it",
    local_dir="weights/gemma-4-E4B-it",
    ignore_patterns=["*.bin"],
)
print("Gemma done.")

print("Downloading BAAI/bge-m3 (embedder)...")
snapshot_download(
    repo_id="BAAI/bge-m3",
    local_dir="weights/bge-m3",
)
print("BGE-M3 done.")

## Step 5 — Configure pipeline to use vLLM and local weight paths

In [ ]:
import yaml, os

cfg_path = "configs/pipeline_config.yaml"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Switch to vLLM backend
cfg["backend"] = "vllm"

# Point to local weight directories
cfg["models"]["primary"]   = os.path.abspath("weights/Qwen3.5-9B")
cfg["models"]["secondary"] = os.path.abspath("weights/gemma-4-E4B-it")
cfg["models"]["embedder"]  = os.path.abspath("weights/bge-m3")

# Tune for available VRAM (A100 = 40 GB or 80 GB)
import subprocess
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True
)
vram_mb = int(result.stdout.strip().split("\n")[0])
print(f"Detected VRAM: {vram_mb} MB")

# 40 GB A100: one model at a time fits; 80 GB: both fit simultaneously
cfg["vllm"]["gpu_memory_utilization"] = 0.92
cfg["vllm"]["max_model_len"] = 16384 if vram_mb >= 38000 else 8192
cfg["vllm"]["dtype"] = "bfloat16"
cfg["vllm"]["tensor_parallel_size"] = 1

with open(cfg_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

print("Config updated:")
print(f"  backend          = {cfg['backend']}")
print(f"  primary model    = {cfg['models']['primary']}")
print(f"  secondary model  = {cfg['models']['secondary']}")
print(f"  max_model_len    = {cfg['vllm']['max_model_len']}")

## Step 6 — (Optional) Build RAG index

Skip if `faiss_narrow.index` already exists or RAG is disabled.

In [ ]:
import os
if not os.path.exists("faiss_narrow.index"):
    print("Building RAG index...")
    !python scripts/build_index.py
    print("Index built.")
else:
    print("Index already present, skipping.")

## Step 7 — Run the full pipeline

Replace `data/questions.csv` with the actual competition input file path.

In [ ]:
INPUT_FILE  = "data/questions.csv"   # <-- competition input
OUTPUT_FILE = "submission.csv"
AUDIT_FILE  = "audit_log.json"

import sys
sys.path.insert(0, ".")

from src.pipeline import run_pipeline

run_pipeline(
    input_path=INPUT_FILE,
    output_path=OUTPUT_FILE,
    audit_path=AUDIT_FILE,
    strict=False,
)
print("Pipeline complete.")

## Step 8 — Validate & score the submission

In [ ]:
import pandas as pd

sub = pd.read_csv(OUTPUT_FILE)
print(f"Rows: {len(sub)}")
print(sub.head(10))

# Check invariants
assert sub["id"].is_unique, "Duplicate IDs!"
assert sub["answer"].notna().all(), "Missing answers!"
print("All invariants passed.")

## Step 9 — (Optional) Evaluate on dev set

In [ ]:
!python eval/score.py \
    --pred submission.csv \
    --gold eval/dev_set.jsonl \
    --audit audit_log.json

## Tips & Troubleshooting

| Symptom | Fix |
|---|---|
| `CUDA out of memory` | Reduce `gpu_memory_utilization` to 0.85, or set `secondary: ""` to skip Gemma |
| `VllmBackend requires messages` | You're calling a legacy code path — update to `_make_request()` |
| Slow inference | Set `tensor_parallel_size: 2` if you have 2 GPUs |
| Weight download fails | Check `HF_TOKEN` and that you accepted the model license on huggingface.co |
| Gemma model not found | Use `google/gemma-3-4b-it` as a drop-in fallback in the config |

### Memory guide (A100 40 GB)
- Qwen3.5-9B bfloat16: ~18 GB
- Gemma 4 E4B bfloat16: ~9 GB
- BGE-M3: ~2 GB
- KV cache (16384 ctx): ~6–8 GB
- **Total: ~35–37 GB** — fits on 40 GB with `gpu_memory_utilization=0.92`

### Switching between local (llamacpp) and Linux (vllm)
Only one line in `configs/pipeline_config.yaml` needs to change:
```yaml
backend: llamacpp   # MacBook
backend: vllm       # Linux/Colab
```